# Music Annotation Project V9.0A
## Mask-Authority Transfer Audit using the actual Notebook 53R3 execution cells

**Runtime:** CPU. Run all in a fresh Colab runtime.

This notebook does not search for a Python package and does not invent a V8.8 API. The execution cells below preserve the actual Notebook 53R3 mechanics:

- full-page regional `M3.png` masks are unioned without converting region boxes into masks;
- Notebook 50 decision canvases are restored origin-aligned to the physical page;
- SAFE_REMOVE is decision code 1 and SAFE_PROTECT is decision code 2;
- the operational V8.8 route is `M5_REMOVE = M3 & ~SAFE_REMOVE`;
- reconstruction uses full-page OpenCV TELEA with radius 3 on each regional `M5_REMOVE` mask;
- the candidate starts from the frozen V7.8 base, restores the frozen protected-print bytes, and replaces only regional M5 pixels;
- outside-authority bytes are asserted immutable.

### Important scope correction
Notebook 53R3 does **not** derive M3 from an arbitrary YOLO or reviewed seed mask. It consumes M3 regions produced upstream by the Notebook 45/46 lineage. Therefore V9.0A cannot honestly substitute a raw YOLO mask directly into Notebook 53R3 and call the result V8.8.

Each mask-authority arm must provide a complete, arm-specific upstream package containing the observed page, frozen V7.8 base, protected-print mask, regional M3 masks, and Notebook 50 decision map. This notebook compares those packages through the exact Notebook 53R3 operational REMOVE-only cells. It fails closed if an arm package is missing or ambiguous.

Raw YOLO remains proposal evidence only. This notebook does not authorise automatic mask authority or SAFE_PROTECT.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib


## 1. Configuration and explicit arm-package contract

No generic `/v8_8/` directory is assumed. The reviewed comparator defaults to the actual Notebook 53R3 upstream roots. Candidate mask arms must be listed in an explicit CSV because the supplied references do not establish an executable upstream builder that converts an arbitrary seed mask into Notebook 45/46 M3 artefacts.

Create `MusicAnnotationProject/v9_0a_arm_packages.csv` with one row per arm and these columns:

- `arm`
- `n45_run`
- `n46_run`
- `n50_run`
- `source_v82_run`
- `role` (`reviewed_comparator`, `raw_yolo`, `confidence_gated_yolo`, or `reviewer_corrected_yolo`)
- `detector_or_mask_identity`
- `authority_status`

Every row must point to a completed, arm-specific package. Reusing the reviewed M3 package under a different arm label is rejected.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
N45_ROOT = ROOT / 'observed_print_classifier_v8_6_notebook45'
N46_ROOT = ROOT / 'observed_print_classifier_v8_6_notebook46'
N50_ROOT = ROOT / 'observable_residual_mask_m5_v8_7'
N51_ROOT = ROOT / 'observed_print_classifier_v8_7_notebook51'
N53_ROOT = ROOT / 'observed_print_classifier_v8_8_notebook53R3'
ARM_MANIFEST_PATH = ROOT / 'v9_0a_arm_packages.csv'
OUTPUT_ROOT = ROOT / 'observed_print_classifier_v9_0a_mask_authority_transfer'

# Exact overrides are optional, but the definitive run should pin them.
REFERENCE_N53_RUN_OVERRIDE = None
REFERENCE_N51_RUN_OVERRIDE = None
PAGE_ALLOWLIST = None
INPAINT_RADIUS = 3
SAFE_REMOVE_CODE = 1
SAFE_PROTECT_CODE = 2

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v9_0a_53R3_mask_authority_transfer'
OUT = OUTPUT_ROOT / RUN_ID
INPUTS = OUT / 'inputs'; PAGES_OUT = OUT / 'pages'; TABLES = OUT / 'tables'; AUDIT = OUT / 'audit'; VISUAL = OUT / 'visual_audit'; UPLOAD = OUT / 'upload'
for p in [OUT, INPUTS, PAGES_OUT, TABLES, AUDIT, VISUAL, UPLOAD]: p.mkdir(parents=True, exist_ok=False)

def require(condition, message):
    if not condition:
        try:
            (AUDIT/'fatal_failure.json').write_text(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),'message':str(message)},indent=2))
        except Exception:
            pass
        raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def newest_complete(root, required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

print('Output:',OUT)


## 2. Resolve the passed Notebook 53R3 and Notebook 51R2 references

The reference run is used for lineage and exact reviewed-arm regression checks. It is not treated as a reusable Python library.


In [ ]:
REQ53=['stage_f_verdict.json','experiment_identity.json','tables/page_equivalence.csv','pages']
REQ51=['04_candidate_lock/candidate_inventory.csv','04_candidate_lock/candidate_lock.json']
N53=Path(REFERENCE_N53_RUN_OVERRIDE) if REFERENCE_N53_RUN_OVERRIDE else newest_complete(N53_ROOT,REQ53)
N51=Path(REFERENCE_N51_RUN_OVERRIDE) if REFERENCE_N51_RUN_OVERRIDE else newest_complete(N51_ROOT,REQ51)
require(N53 is not None,'No complete Notebook53R3 run found. Set REFERENCE_N53_RUN_OVERRIDE.')
require(N51 is not None,'No complete Notebook51R2 run found. Set REFERENCE_N51_RUN_OVERRIDE.')
verdict53=json.loads((N53/'stage_f_verdict.json').read_text()); identity53=json.loads((N53/'experiment_identity.json').read_text())
require(identity53.get('version')=='53R3-v8.8b-notebook51R2-regression-compatibility','Unexpected Notebook53R3 version.')
require(identity53.get('mode')=='REGISTERED_REPLAY','Selected Notebook53R3 is not the passed registered-replay lineage.')
require(verdict53.get('stage_F_corrected_page_exact') is True,'Notebook53R3 Stage F did not pass exact corrected-page compatibility.')
require(identity53.get('operational_safe_protect_authorised') is False,'Unexpected SAFE_PROTECT authorisation state.')
print('Notebook53R3:',N53)
print('Notebook51R2:',N51)


## 3. Load and validate explicit arm packages

This is deliberately stricter than selecting the newest matching folder. A labelled arm is accepted only when the manifest points to complete artefacts and the package has a distinct semantic identity.


In [ ]:
REQUIRED_COLUMNS={'arm','n45_run','n46_run','n50_run','source_v82_run','role','detector_or_mask_identity','authority_status'}
require(ARM_MANIFEST_PATH.exists(),f'Missing explicit arm manifest: {ARM_MANIFEST_PATH}')
arms=pd.read_csv(ARM_MANIFEST_PATH,dtype=str,keep_default_na=False)
require(REQUIRED_COLUMNS.issubset(arms.columns),f'Arm manifest missing {sorted(REQUIRED_COLUMNS-set(arms.columns))}')
require(not arms.arm.duplicated().any(),'Arm names must be unique.')
require((arms.arm.str.strip()!='').all(),'Blank arm name.')
require((arms.detector_or_mask_identity.str.strip()!='').all(),'Every arm requires detector_or_mask_identity.')
require((arms.authority_status.str.strip()!='').all(),'Every arm requires authority_status.')
require((arms.role=='reviewed_comparator').sum()==1,'Exactly one reviewed_comparator arm is required.')
require((arms.detector_or_mask_identity.nunique()==len(arms)),'Arm identities must be distinct. Do not relabel the same upstream package as multiple arms.')

REQ45=['mask_lock.json','source_record.json','tables/region_manifest.parquet']
REQ46=['source_record.json','audit/output_completion.json']
REQ50=['notebook50_verdict.json','tables/m5_map_inventory.csv']
validated=[]
for _,r in arms.iterrows():
    n45=Path(r.n45_run); n46=Path(r.n46_run); n50=Path(r.n50_run); sv82=Path(r.source_v82_run)
    missing=[]
    for root,req,label in [(n45,REQ45,'N45'),(n46,REQ46,'N46'),(n50,REQ50,'N50')]:
        missing += [f'{label}:{x}' for x in req if not (root/x).exists()]
    require(not missing,f'{r.arm}: incomplete arm package: {missing}')
    require(sv82.exists(),f'{r.arm}: source_v82_run does not exist: {sv82}')
    validated.append({'arm':r.arm,'role':r.role,'n45_run':str(n45),'n46_run':str(n46),'n50_run':str(n50),'source_v82_run':str(sv82),'detector_or_mask_identity':r.detector_or_mask_identity,'authority_status':r.authority_status,'n45_mask_lock_sha256':sha256_file(n45/'mask_lock.json'),'n45_region_manifest_sha256':sha256_file(n45/'tables/region_manifest.parquet'),'n50_inventory_sha256':sha256_file(n50/'tables/m5_map_inventory.csv')})
validated_df=pd.DataFrame(validated)
# Reject packages that are operationally identical despite different labels.
sigcols=['n45_mask_lock_sha256','n45_region_manifest_sha256','n50_inventory_sha256']
require(not validated_df.duplicated(sigcols).any(),'Two or more arms have identical N45/N50 semantic package hashes. This would not test mask-authority transfer.')
validated_df.to_csv(INPUTS/'validated_arm_packages.csv',index=False)
display(validated_df)


## 4. Actual Notebook 53R3 helper cells

The following functions are copied from the Notebook 53R3 execution path. The only structural change is parameterisation by arm package instead of global `N45`, `N46` and `N50` variables.


In [ ]:
def load_arm_package(row):
    N45=Path(row.n45_run); N46=Path(row.n46_run); N50=Path(row.n50_run); source_v82=Path(row.source_v82_run)
    regions=pd.read_parquet(N45/'tables/region_manifest.parquet')
    need={'page','source_group','region_id','y0','y1','x0','x1'}
    require(need.issubset(regions.columns),f'{row.arm}: Notebook45 region manifest missing {sorted(need-set(regions.columns))}')
    regions['page']=regions.page.astype(str); regions['source_group']=regions.source_group.astype(str)
    require(not regions[['page','region_id']].duplicated().any(),f'{row.arm}: duplicate Notebook45 region identity.')
    inventory=pd.read_csv(N50/'tables/m5_map_inventory.csv',dtype=str,keep_default_na=False)
    require({'page','file'}.issubset(inventory.columns),f'{row.arm}: Notebook50 inventory lacks page/file.')
    inventory['page']=inventory.page.astype(str)
    return {'N45':N45,'N46':N46,'N50':N50,'source_v82':source_v82,'regions':regions,'inventory':inventory}

def resolve_base(pkg,page):
    candidates=[pkg['N46']/'whole_page_candidates'/page/'v7_8.png',pkg['source_v82']/'whole_pages'/page/'v7_8.png']
    for p in candidates:
        if p.exists(): return p
    raise RuntimeError(f'{page}: cannot resolve frozen V7.8 base.')

def resolve_protected(pkg,page):
    p=pkg['N46']/'whole_page_candidates'/page/'protected_print.png'
    require(p.exists(),f'{page}: missing protected_print.png: {p}')
    return p

def union_arm(pkg,page,arm='M3'):
    arr=None
    for _,r in pkg['regions'][pkg['regions'].page==page].iterrows():
        p=pkg['N45']/'masks'/page/str(r.region_id).replace(':','_')/f'{arm}.png'
        m=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
        require(m is not None,f'Missing/unreadable Notebook45 mask: {p}')
        b=m>0; arr=b.copy() if arr is None else (arr|b)
    require(arr is not None,f'{page}: no regional {arm} masks.')
    return arr

def resolve_notebook50_decisions(pkg,page,page_shape):
    q=pkg['inventory'][pkg['inventory'].page==page]
    require(len(q)==1,f'{page}: expected exactly one Notebook50 M5 inventory row, found {len(q)}.')
    p=Path(q.iloc[0]['file'])
    if not p.exists():
        hits=[x for x in pkg['N50'].rglob(p.name) if x.is_file()]
        require(len(hits)==1,f'{page}: could not uniquely resolve Notebook50 decision file {p.name}: {hits}')
        p=hits[0]
    require(p.suffix.lower()=='.npz',f'{page}: expected Notebook50 decision NPZ, got {p}')
    with np.load(p,allow_pickle=False) as z:
        require('m5_decision_map' in z.files,f'{page}: {p.name} lacks m5_decision_map')
        small=np.asarray(z['m5_decision_map'])
    require(small.ndim==2,f'{page}: decision map is not 2-D: {small.shape}')
    require(small.shape[0]<=page_shape[0] and small.shape[1]<=page_shape[1],f'{page}: decision map {small.shape} exceeds page {page_shape}')
    decision=np.zeros(page_shape,dtype=small.dtype); decision[:small.shape[0],:small.shape[1]]=small
    return decision==SAFE_REMOVE_CODE,decision==SAFE_PROTECT_CODE,p,small.shape

def reconstruct_telea_exact(obs,mask):
    m=(mask>0).astype(np.uint8)*255
    if not m.any(): return obs.copy()
    cand=cv2.inpaint(obs,m,INPAINT_RADIUS,cv2.INPAINT_TELEA)
    out=obs.copy(); out[m>0]=cand[m>0]
    require(np.array_equal(out[m==0],obs[m==0]),'Telea outside-mask drift.')
    return out


## 5. Exact operational REMOVE-only execution by arm

This cell deliberately uses `M5_REMOVE = M3 & ~SAFE_REMOVE`. It records SAFE_PROTECT decisions but does not exclude them from the reconstruction target. The historical `REGISTERED_REPLAY` route is not used as the operational policy.


In [ ]:
runtime={}; mask_rows=[]; page_rows=[]; region_rows=[]
arm_rosters={}
for _,row in validated_df.iterrows():
    arm=row.arm; pkg=load_arm_package(row); pages=sorted(pkg['regions'].page.unique())
    if PAGE_ALLOWLIST is not None: pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
    require(pages,f'{arm}: empty page roster.')
    arm_rosters[arm]=set(pages); runtime[arm]={}
    for page in pages:
        obs_path=pkg['source_v82']/'whole_pages'/page/'observed.png'; base_path=resolve_base(pkg,page); protected_path=resolve_protected(pkg,page)
        obs=cv2.imread(str(obs_path),cv2.IMREAD_GRAYSCALE); base=cv2.imread(str(base_path),cv2.IMREAD_GRAYSCALE); protected=cv2.imread(str(protected_path),cv2.IMREAD_GRAYSCALE)
        m3=union_arm(pkg,page,'M3')
        require(obs is not None and base is not None and protected is not None,f'{arm}/{page}: missing page inputs.')
        require(obs.shape==base.shape==protected.shape==m3.shape,f'{arm}/{page}: page input shape mismatch.')
        sr,sp,decision_path,small_shape=resolve_notebook50_decisions(pkg,page,m3.shape)
        require(not np.any(sr&~m3),f'{arm}/{page}: SAFE_REMOVE outside M3')
        require(not np.any(sp&~m3),f'{arm}/{page}: SAFE_PROTECT outside M3')
        require(not np.any(sr&sp),f'{arm}/{page}: SAFE_REMOVE/SAFE_PROTECT overlap')
        m5=m3&~sr
        candidate=base.copy(); candidate[protected>0]=obs[protected>0]
        regional_union=np.zeros_like(m5,dtype=bool)
        for _,r in pkg['regions'][pkg['regions'].page==page].iterrows():
            rid=str(r.region_id); mp=pkg['N45']/'masks'/page/rid.replace(':','_')/'M3.png'; rm=cv2.imread(str(mp),cv2.IMREAD_GRAYSCALE)
            require(rm is not None and rm.shape==obs.shape,f'{arm}/{page}/{rid}: invalid regional M3 {mp}')
            region_m3=rm>0; regional=m5&region_m3; regional_union|=regional
            if regional.any():
                recon=reconstruct_telea_exact(obs,regional); candidate[regional]=recon[regional]
            region_rows.append({'arm':arm,'page':page,'region_id':rid,'regional_M3_pixels':int(region_m3.sum()),'regional_M5_REMOVE_pixels':int(regional.sum()),'regional_mask_sha256':array_hash(regional.astype(np.uint8)),'m3_path':str(mp)})
        require(np.array_equal(regional_union,m5),f'{arm}/{page}: regional M5 union differs from page M5')
        outside_authority=int(np.sum(candidate[~m3]!=base[~m3]))
        require(outside_authority==0,f'{arm}/{page}: candidate changed frozen base outside M3 authority: {outside_authority}')
        d=PAGES_OUT/arm/page; d.mkdir(parents=True,exist_ok=False)
        for name,a in [('observed',obs),('base_v7_8',base),('protected_print',protected),('m3',m3.astype(np.uint8)*255),('safe_remove',sr.astype(np.uint8)*255),('safe_protect_decision',sp.astype(np.uint8)*255),('m5_remove',m5.astype(np.uint8)*255),('corrected',candidate)]: cv2.imwrite(str(d/f'{name}.png'),a)
        runtime[arm][page]={'obs':obs,'base':base,'protected':protected>0,'m3':m3,'sr':sr,'sp':sp,'m5':m5,'corrected':candidate}
        mask_rows.append({'arm':arm,'page':page,'M3_pixels':int(m3.sum()),'SAFE_REMOVE_pixels':int(sr.sum()),'SAFE_PROTECT_decision_pixels':int(sp.sum()),'M5_REMOVE_pixels':int(m5.sum()),'decision_canvas_shape':str(tuple(small_shape)),'page_shape':str(tuple(m3.shape)),'decision_path':str(decision_path),'decision_sha256':sha256_file(decision_path)})
        page_rows.append({'arm':arm,'page':page,'outside_M3_base_drift':outside_authority,'observed_array_sha256':array_hash(obs),'base_array_sha256':array_hash(base),'m3_array_sha256':array_hash(m3.astype(np.uint8)),'safe_remove_array_sha256':array_hash(sr.astype(np.uint8)),'m5_remove_array_sha256':array_hash(m5.astype(np.uint8)),'corrected_array_sha256':array_hash(candidate)})

rosters=list(arm_rosters.values()); require(all(r==rosters[0] for r in rosters[1:]),'Arm page rosters differ. Paired comparison requires exact roster equality.')
mask_df=pd.DataFrame(mask_rows); page_df=pd.DataFrame(page_rows); region_df=pd.DataFrame(region_rows)
mask_df.to_csv(TABLES/'arm_mask_inventory.csv',index=False); page_df.to_csv(AUDIT/'arm_output_integrity.csv',index=False); region_df.to_csv(TABLES/'arm_regional_masks.csv',index=False)
print('Arms:',sorted(runtime)); print('Paired pages:',len(rosters[0]))


## 6. Reviewed-arm exact-regression gate

The reviewed comparator must reproduce the selected Notebook 53R3 page state at the shared M3, SAFE_REMOVE and base arrays. Its operational REMOVE-only corrected page is not required to equal Notebook 53R3's historical registered-replay corrected page because Notebook 53R3 expressly documents the code-2 lineage inconsistency.


In [ ]:
review_arm=validated_df.loc[validated_df.role=='reviewed_comparator','arm'].iloc[0]
review_checks=[]
for page,z in runtime[review_arm].items():
    refdir=N53/'pages'/page
    require(refdir.exists(),f'{page}: selected Notebook53R3 reference page missing.')
    refs={k:cv2.imread(str(refdir/f'{fn}.png'),cv2.IMREAD_GRAYSCALE) for k,fn in [('obs','observed'),('m3','m3'),('sr','safe_remove')]}
    require(all(v is not None for v in refs.values()),f'{page}: incomplete Notebook53R3 reference arrays.')
    review_checks.append({'page':page,'observed_exact':bool(np.array_equal(z['obs'],refs['obs'])),'M3_exact':bool(np.array_equal(z['m3'],refs['m3']>0)),'SAFE_REMOVE_exact':bool(np.array_equal(z['sr'],refs['sr']>0))})
review_gate=pd.DataFrame(review_checks); review_gate.to_csv(AUDIT/'reviewed_arm_53R3_input_equivalence.csv',index=False)
require(review_gate[['observed_exact','M3_exact','SAFE_REMOVE_exact']].all().all(),'Reviewed comparator does not reproduce the selected Notebook53R3 input state.')
display(review_gate)


## 7. Prediction lock before comparative interpretation


In [ ]:
prediction_lock={'created_utc':datetime.now(timezone.utc).isoformat(),'notebook53R3_reference':str(N53),'notebook53R3_identity_sha256':sha256_file(N53/'experiment_identity.json'),'notebook53R3_stage_f_verdict_sha256':sha256_file(N53/'stage_f_verdict.json'),'arm_manifest_sha256':sha256_file(ARM_MANIFEST_PATH),'validated_arm_packages_sha256':sha256_file(INPUTS/'validated_arm_packages.csv'),'arm_output_integrity_sha256':sha256_file(AUDIT/'arm_output_integrity.csv'),'reviewed_arm_53R3_input_equivalence_sha256':sha256_file(AUDIT/'reviewed_arm_53R3_input_equivalence.csv'),'mode':'OPERATIONAL_REMOVE_ONLY','safe_protect_operationally_authorised':False,'outputs':page_df.sort_values(['arm','page'])[['arm','page','m3_array_sha256','safe_remove_array_sha256','m5_remove_array_sha256','corrected_array_sha256']].to_dict('records')}
(AUDIT/'prediction_lock.json').write_text(json.dumps(prediction_lock,indent=2))
print('Prediction lock:',sha256_file(AUDIT/'prediction_lock.json'))


## 8. Paired mask-authority transfer evidence

Comparisons are arm-to-reviewed-comparator effects. They are not claims that the reviewed mask is perfect truth.


In [ ]:
paired=[]
for arm,pageset in runtime.items():
    if arm==review_arm: continue
    for page,z in pageset.items():
        ref=runtime[review_arm][page]
        m3xor=z['m3']^ref['m3']; srxor=z['sr']^ref['sr']; m5xor=z['m5']^ref['m5']; outxor=z['corrected']!=ref['corrected']
        paired.append({'arm':arm,'page':page,'M3_xor_pixels':int(m3xor.sum()),'SAFE_REMOVE_xor_pixels':int(srxor.sum()),'M5_REMOVE_xor_pixels':int(m5xor.sum()),'corrected_xor_pixels':int(outxor.sum()),'candidate_M3_overreach_vs_reviewed':int((z['m3']&~ref['m3']).sum()),'candidate_M3_missed_vs_reviewed':int((ref['m3']&~z['m3']).sum()),'candidate_SAFE_REMOVE_overreach_vs_reviewed':int((z['sr']&~ref['sr']).sum()),'candidate_SAFE_REMOVE_missed_vs_reviewed':int((ref['sr']&~z['sr']).sum())})
paired_df=pd.DataFrame(paired); paired_df.to_csv(TABLES/'paired_transfer_evidence.csv',index=False)
summary=paired_df.groupby('arm',as_index=False).agg(pages=('page','nunique'),M3_xor_pixels=('M3_xor_pixels','sum'),SAFE_REMOVE_xor_pixels=('SAFE_REMOVE_xor_pixels','sum'),M5_REMOVE_xor_pixels=('M5_REMOVE_xor_pixels','sum'),corrected_xor_pixels=('corrected_xor_pixels','sum'),M3_overreach=('candidate_M3_overreach_vs_reviewed','sum'),M3_missed=('candidate_M3_missed_vs_reviewed','sum'))
summary.to_csv(TABLES/'arm_summary.csv',index=False); display(summary)


## 9. Visual audit queue


In [ ]:
queue=paired_df.assign(review_priority=lambda x:x.corrected_xor_pixels+5*x.candidate_M3_overreach_vs_reviewed+3*x.candidate_M3_missed_vs_reviewed).sort_values('review_priority',ascending=False)
queue.to_csv(TABLES/'visual_review_queue.csv',index=False)
for i,r in queue.head(30).reset_index(drop=True).iterrows():
    page=r.page; arm=r.arm; ref=runtime[review_arm][page]; cand=runtime[arm][page]
    overlay=cv2.cvtColor(ref['obs'],cv2.COLOR_GRAY2RGB); overlay[ref['m3']&cand['m3']]=(0,180,0); overlay[ref['m3']&~cand['m3']]=(255,165,0); overlay[cand['m3']&~ref['m3']]=(255,0,0)
    fig,ax=plt.subplots(1,4,figsize=(20,7))
    for a,im,title in zip(ax,[ref['obs'],overlay,ref['corrected'],cand['corrected']],['Observed','M3 overlap green / missed orange / overreach red','Reviewed comparator','Candidate '+arm]): a.imshow(im,cmap='gray' if im.ndim==2 else None); a.set_title(title); a.axis('off')
    fig.suptitle(f'{page} | {arm}'); plt.tight_layout(); fig.savefig(VISUAL/f'{i+1:03d}_{page}_{arm}.png',dpi=130,bbox_inches='tight'); plt.close(fig)
print('Visual queue:',VISUAL)


## 10. Fail-closed verdict

V9.0A reports whether complete upstream mask-authority packages can be compared through the exact Notebook 53R3 operational REMOVE-only execution path. It does not create an automatic mask-authority rule.


In [ ]:
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0A-53R3-cell-grounded-mask-authority-transfer','development_shadow_only':True,'automatic_mask_authority_authorised':False,'notebook53R3_cells_used':True,'python_module_discovery_used':False,'mode':'OPERATIONAL_REMOVE_ONLY','safe_protect_operationally_authorised':False,'reviewed_comparator':review_arm,'arms':validated_df[['arm','role','detector_or_mask_identity','authority_status']].to_dict('records'),'paired_pages':len(rosters[0]),'reviewed_arm_53R3_input_equivalence_passed':True,'outside_M3_base_drift_zero':bool((page_df.outside_M3_base_drift==0).all()),'prediction_lock_sha256':sha256_file(AUDIT/'prediction_lock.json'),'interpretation':'Paired evidence only. Each arm was required to provide a complete upstream Notebook45/46/50 package. No raw seed mask was treated as a valid substitute for M3.'}
(OUT/'v9_0a_verdict.json').write_text(json.dumps(verdict,indent=2)); print(json.dumps(verdict,indent=2))


## 11. Final Upload Package


In [ ]:
import zipfile
required=[OUT/'v9_0a_verdict.json',INPUTS/'validated_arm_packages.csv',AUDIT/'prediction_lock.json',AUDIT/'arm_output_integrity.csv',AUDIT/'reviewed_arm_53R3_input_equivalence.csv',TABLES/'arm_mask_inventory.csv',TABLES/'arm_regional_masks.csv',TABLES/'paired_transfer_evidence.csv',TABLES/'arm_summary.csv',TABLES/'visual_review_queue.csv']
for p in required: require(p.exists(),f'Missing upload artefact: {p}')
(UPLOAD/'README.txt').write_text('V9.0A MASK-AUTHORITY TRANSFER AUDIT
Uses actual Notebook53R3 execution cells in OPERATIONAL_REMOVE_ONLY mode.
Requires complete arm-specific Notebook45/46/50 upstream packages.
No automatic mask authority is authorised.
')
rows=[]
for src in required+sorted(VISUAL.glob('*.png'))[:30]:
    rel=('visuals/'+src.name) if src.suffix.lower()=='.png' else ('tables/'+src.name if src.parent==TABLES else 'audit/'+src.name if src.parent==AUDIT else 'inputs/'+src.name if src.parent==INPUTS else src.name)
    dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'upload_file_list.txt').write_text('
'.join(x['relative_path'] for x in rows))
zip_path=UPLOAD/'v9_0a_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.rglob('*'):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
require(zip_path.exists(),'Upload bundle creation failed.'); print('Upload bundle:',zip_path)


## Required upstream question

If raw-YOLO and confidence-gated arms do not yet have complete Notebook 45/46/50 packages, this notebook will stop. The missing implementation is not in Notebook 53R3: it is the verified upstream builder that converts each alternative authoritative seed mask into the exact M3 regions, protected-print page and Notebook 50 decision map expected by Notebook 53R3. That builder must be identified from the relevant upstream notebooks before those arms can be executed without guessing.
